### Задание 1
Счётчик

In [13]:
class Counter:
    def __init__(self, start: int = 0, stop: int = 100):
        if start > stop:
            raise ValueError("Начальное значение не может быть больше верхней границы")
        self._start = start
        self._stop = stop
        self._current = start

    def set_value(self, value: int):
        if not (self._start <= value <= self._stop):
            raise ValueError("Значение вне допустимого диапазона")
        self._current = value

    def incr(self, delta: int = 1):
        new_value = self._current + delta
        if new_value > self._stop:
            raise ValueError("Увеличение приведет к превышению верхней границы")
        self._current = new_value

    def decr(self, delta: int = 1):
        new_value = self._current - delta
        if new_value < self._start:
            raise ValueError("Уменьшение приведет к выходу за нижнюю границу")
        self._current = new_value

    def get(self):
        return self._current

    def stop(self):
        self._current = self._start

    def __add__(self, other: 'Counter'):
        new_stop = self._stop + other._stop
        new_start = 0
        new_current = self._current + other._current
        if new_current > new_stop:
            raise ValueError("Сумма значений превышает новую верхнюю границу")

        result = Counter(new_start, new_stop)
        result._current = new_current
        return result

In [14]:
c1 = Counter(10, 20)
c2 = Counter(15, 30)
c1.incr()
c2.set_value(25)
c2.decr(2)
c3 = c1 + c2
print(c3.get())
c3.stop()
print(c3.get())

34
0


In [15]:
while True:
    c3.incr(2)

ValueError: Увеличение приведет к превышению верхней границы

In [16]:
c3.get()

50

### Задание 2


In [17]:
from typing import Protocol
import math


class Rectangle:
    def __init__(self, a: float, b: float) -> None:
        self.a: float = a
        self.b: float = b

    def get_perimeter(self) -> float:
        return 2 * (self.a + self.b)

    def get_square(self) -> float:
        return self.a * self.b


class Circle:   
    def __init__(self, r: float) -> None:
        self.r: float = r

    def get_perimeter(self) -> float:
        return 2 * math.pi * self.r

    def get_square(self) -> float:
        return math.pi * self.r ** 2


class Rhombus:
    def __init__(self, p: float, q: float) -> None:
        self.p: float = p
        self.q: float = q

    def get_perimeter(self) -> float:
        side: float = math.sqrt((self.p / 2) ** 2 + (self.q / 2) ** 2)
        return 4 * side

    def get_square(self) -> float:
        return (self.p * self.q) / 2

class Figures(Protocol):
    def get_perimeter(self) -> float:
        ...
    def get_square(self) -> float:
        ...


def calculate_perimeter(figure: Figures) -> float:
    return figure.get_perimeter()

def calculate_square(figure: Figures) -> float:
    return figure.get_square()

In [18]:
rec = Rectangle(2.0, 3.0)
rec.get_perimeter(), rec.get_square()

(10.0, 6.0)

In [19]:
cir = Circle(3.0)
cir.get_perimeter(), cir.get_square()

(18.84955592153876, 28.274333882308138)

In [20]:
rhom = Rhombus(3.0, 4.0)
rhom.get_perimeter(), rhom.get_square()

(10.0, 6.0)

In [21]:
calculate_perimeter(rec), calculate_perimeter(cir), calculate_perimeter(rhom)

(10.0, 18.84955592153876, 10.0)

In [22]:
calculate_square(rec), calculate_square(cir), calculate_square(rhom)

(6.0, 28.274333882308138, 6.0)

### Задание 3
Задание: Система бронирования коворкингов

In [23]:
import datetime
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional, List

# 1. Композиция: местоположение
@dataclass
class Address:
    city: str
    street: str
    floor: int

# 2. Композиция: пользователь
@dataclass
class User:
    id: int
    name: str
    rating: float  # рейтинг от 1 до 5
    bonus_points: int  # бонусные баллы для компенсации

class WorkspaceType(Enum):
    DESK = "рабочее место"
    PRIVATE_OFFICE = "личный кабинет"
    MEETING_ROOM = "переговорная"

class BookingStatus(Enum):
    ACTIVE = "активно"
    CANCELLED = "отменено"

_booking_id_counter = 0

def _generate_booking_id() -> int:
    global _booking_id_counter
    _booking_id_counter += 1
    return _booking_id_counter

@dataclass
class Booking:
    """
    Бронирование рабочего места
    """
    # Композиция: бронирование имеет адрес и пользователя
    address: Address
    user: User
    workspace_type: WorkspaceType
    price_per_hour: int
    date: datetime.date
    hours: int  # количество часов

    # TODO: Добавьте поле для статуса бронирования (активно/отменено)
    status: BookingStatus = BookingStatus.ACTIVE

    # TODO: Добавьте поле id (автоматически)
    id: int = field(default_factory=_generate_booking_id)
    
    # TODO: Добавьте поле created_at (время создания)
    created_at: datetime.datetime = field(default_factory=datetime.datetime.now)

    def __post_init__(self) -> None:
        if self.hours <= 0:
            raise ValueError("Количество часов должно быть положительным")
        if not (1.0 <= self.user.rating <= 5.0):
            raise ValueError("Рейтинг пользователя должен быть от 1 до 5")
        if self.price_per_hour <= 0:
            raise ValueError("Цена за час должна быть положительной")

    # TODO: Создайте метод total_price(), возвращающий общую стоимость
    def total_price(self) -> int:
        return self.price_per_hour * self.hours

    # TODO: Создайте метод cancel(), меняющий статус
    def cancel(self) -> None:
        if self.status == BookingStatus.ACTIVE:
            self.status = BookingStatus.CANCELLED
            print(f"Бронирование {self.id} отменено.")
        else:
            print(f"Бронирование {self.id} уже отменено.")

    # TODO: Создайте метод change_user(new_user), передающий бронь другому пользователю
    def change_user(self, new_user: User) -> bool:
        if self.status != BookingStatus.ACTIVE:
            print("Нельзя передать отмененное бронирование.")
            return False

        old_user = self.user
        self.user = new_user
        print(f"Бронирование {self.id} передано от {old_user.name} к {new_user.name}.")
        return True


# Функция для обмена бронями между пользователями
def exchange_bookings(booking1: Booking, booking2: Booking) -> bool:
    """
    Обменивает брони между пользователями.
    Если цены разные, разница списывается/начисляется бонусными баллами.
    """
    # 1. Проверить, что обе брони активны
    if booking1.status != BookingStatus.ACTIVE or booking2.status != BookingStatus.ACTIVE:
        print("Одна или обе брони неактивны. Обмен невозможен.")
        return False

    # 2. Проверить, что даты броней совпадают (для простоты)
    if booking1.date != booking2.date:
        print(f"Даты броней не совпадают: {booking1.date} vs {booking2.date}. Обмен невозможен.")
        return False

    # 3. Обменять пользователей местами
    user1 = booking1.user
    user2 = booking2.user

    # 4. Компенсировать разницу в цене бонусными баллами
    price_diff = booking2.total_price() - booking1.total_price()

    if price_diff > 0:
        if user1.bonus_points >= price_diff:
            user1.bonus_points -= price_diff
            user2.bonus_points += price_diff
            print(f"{user1.name} компенсировал разницу в {price_diff} баллов {user2.name}.")
        else:
            print(f"У {user1.name} недостаточно бонусных баллов для компенсации.")
            return False
    elif price_diff < 0:
        price_diff_abs = -price_diff
        if user2.bonus_points >= price_diff_abs:
            user2.bonus_points -= price_diff_abs
            user1.bonus_points += price_diff_abs
            print(f"{user2.name} компенсировал разницу в {price_diff_abs} баллов {user1.name}.")
        else:
            print(f"У {user2.name} недостаточно бонусных баллов для компенсации.")
            return False
    else:
        print("Стоимость броней одинакова, компенсация не требуется.")

    booking1.user, booking2.user = user2, user1
    print(f"Обмен бронями между {user1.name} и {user2.name} успешно завершен.")
    return True

In [24]:
if __name__ == "__main__":
    # Создаем пользователей
    alice = User(id=1, name="Алиса", rating=4.8, bonus_points=500)
    bob = User(id=2, name="Боб", rating=4.2, bonus_points=50000)
    ben = User(id=3, name="Бен", rating=1.0, bonus_points=0)

    
    # Создаем места
    desk = Address(city="Москва", street="Тверская, 15", floor=3)
    office = Address(city="Москва", street="Тверская, 15", floor=5)
    
    # Создаем брони
    booking1 = Booking(
        address=desk,
        user=alice,
        workspace_type=WorkspaceType.DESK,
        price_per_hour=300,
        date=datetime.date(2026, 3, 25),
        hours=4
    )
    
    booking2 = Booking(
        address=office,
        user=bob,
        workspace_type=WorkspaceType.PRIVATE_OFFICE,
        price_per_hour=1200,
        date=datetime.date(2026, 3, 25),
        hours=3
    )

    booking3 = Booking(
        address=office,
        user=ben,
        workspace_type=WorkspaceType.MEETING_ROOM,
        price_per_hour=2000,
        date=datetime.date(2026, 3, 25),
        hours=5
    )
    
    # TODO: Протестировать методы
    # - отмену брони
    # - смену пользователя
    # - обмен бронями с компенсацией

    booking1.cancel()
    booking1.cancel()  # Повторная отмена
    
    booking2.change_user(alice)
    booking2.change_user(bob)
    
    exchange_bookings(booking3, booking2)

Бронирование 1 отменено.
Бронирование 1 уже отменено.
Бронирование 2 передано от Боб к Алиса.
Бронирование 2 передано от Алиса к Боб.
Боб компенсировал разницу в 6400 баллов Бен.
Обмен бронями между Бен и Боб успешно завершен.
